# Универсальная подготовка данных аудитории | Hypothesis Lab

Загрузите XLSX, CSV или TSV и запустите **Среда выполнения → Выполнить все** в Google Colab. Ноутбук определит технический тип каждого поля по его значениям, проверит данные по типу и сохранит исходные названия колонок. Специальные названия полей, банковские категории и продукт Tempo в правилах подготовки не используются.

Каждый файл и каждый лист с таблицей сначала готовится отдельно. Листы без заголовков и пустые листы перечисляются в отчете с причиной пропуска. Таблица `prepared_XX.csv` сохраняет проверяемые исходные значения; `analysis_ready_XX.csv` — отдельная аналитическая копия: неверные значения в числовых/логических/однозначных датах заменяются на пустые только здесь, а варианты регистра категорий приводятся к одному написанию. Неоднозначные даты не угадываются, пропуски не заполняются, выбросы не удаляются.

Если можно определить ID сущности, строится предварительная таблица признаков с одной строкой на ID. Точные копии и конфликтующие записи исключаются **только из этой производной таблицы**, с подсчетом исключений в отчете. Таблицы объединяются по одному и тому же имени ID лишь при сильном совпадении значений. Далее создается отдельная числовая матрица для пробной сегментации: выбирается один агрегат для каждого поля, числовые значения ограничиваются по распределению и масштабируются, подходящие категории кодируются. Суммы по числам не считаются автоматически: баланс или возраст нельзя складывать как расходы.

Автоматическая подготовка не знает бизнес-смысл полей. Для загруженного ранее примера событий ниже приведены явные правила: считать расходы отдельно по типу счета и категории. Они включаются только для точной копии проверенного файла независимо от его имени; для других источников задайте их смысл в `TABLE_SETTINGS`. Матрица — проверяемый черновик для ML, а не доказательство качества сегментов.


## 1. Настройки

Можно оставить настройки как есть и выбрать файлы в окне Colab. По умолчанию рассматриваются **все листы** XLSX с читаемыми заголовками. `SHEETS` ограничивает выбор; `HEADER_ROWS` задает положение заголовка на конкретном листе (с нуля). Когда тип или формат нельзя определить надежно, заполните `TYPE_OVERRIDES` и `DATE_FORMATS` по **исходным именам** колонок. Для разных файлов или листов можно указать собственные настройки в `TABLE_SETTINGS`.

`DAYFIRST = None` оставляет неоднозначные даты вида `02.03.2026` без преобразования, пока вы не зададите формат или порядок дня и месяца. `MISSING_TOKENS` предназначен только для маркеров пропуска, которые вы явно подтвердили. Возможные маркеры показываются в отчете, но сами по себе не заменяются на пустые значения. Для каждого листа можно назначить отдельные `missing_tokens` через `TABLE_SETTINGS`. Время без часового пояса трактуется как `SOURCE_TIMEZONE`; укажите пояс источника, если это не UTC.

Для истории и признаков можно задать `ID_COLUMN`, `TIME_COLUMN` и `AS_OF_DATE`. Если ID и время выбраны эвристически, их роль нужно проверить в отчете; для исторического среза укажите дату явно. В `TABLE_SETTINGS` можно задать разные `id_column`, `time_column`, `date_formats`, `missing_tokens`, `unique_key_columns`, `sum_columns`, `behavior_rollups`, `exclude_source_fields` и `numeric_aggregate_preferences` для конкретных файлов/листов. Для поведенческой суммы сначала явно укажите, какое событие является расходом, какое поле является суммой и по чему разбивать расходы. Без этого движок не предполагает, что любое число можно складывать. Таблица без надежной даты не включается в матрицу, кроме случая явного `allow_timeless_features=True` после проверки периода ее актуальности.

`NORMALIZATION_PARAMS_PATH=None` обучает параметры масштаба и кодирования на текущем файле **только для эксперимента**. Для честного сравнения по кварталам сначала задайте `AS_OF_DATE` и получите `normalization_params.json` на обучающем периоде; при обработке следующих кварталов задайте путь к этому JSON. Тогда используется прежняя шкала, пропуски заполняются прежней медианой, новые категории попадают в группу «прочие». При изменении необходимых колонок выполнение останавливается.

В `EXCLUDE_SOURCE_FIELDS` можно указать исходные названия полей, смысл которых не подходит для сегментации (например, служебные метки кампаний). Остальные поля отбираются по типу и качеству автоматически; в `feature_selection.csv` видно, что вошло в расчет. Исключать поле по смыслу можно только после проверки описания данных.

Ноутбук **сам** ищет среди значений похожие на ID записи поля и отмечает подозрительные повторы одного ID с различным содержимым в одно и то же время. Это предварительная проверка: по данным нельзя гарантированно определить уникальный ключ записи. Если ключ известен из описания источника, необязательный `unique_key_columns` в `TABLE_SETTINGS` позволит проверить его точно. Один ID клиента может повторяться в событиях и сам по себе не является их уникальным ключом.


In [ ]:
import os
import sys
from pathlib import Path

INPUT_PATHS = [os.environ["AUDIENCE_INPUT_PATH"]] if os.getenv("AUDIENCE_INPUT_PATH") else []
SHEETS = {}  # пример: {"data.xlsx": ["Лист2", "Лист3"]}; пусто = все листы с заголовками
HEADER_ROWS = {}  # пример: {("data.xlsx", "Лист2"): 2}, если заголовок в 3-й строке
TYPE_OVERRIDES = {}  # пример: {"код_как_текст": "categorical", "дата_текстом": "datetime"}
DATE_FORMATS = {}  # пример: {"дата_текстом": "%d.%m.%Y"}
MISSING_TOKENS = ()  # пример: ("нет данных", "неизвестно"); "NA" не считается пропуском автоматически
SOURCE_TIMEZONE = "UTC"  # пример: "Europe/Moscow"
DAYFIRST = None  # True = день/месяц; False = месяц/день; None = не угадывать неоднозначные даты
ID_COLUMN = None  # исходное имя ID; None = осторожная эвристика
TIME_COLUMN = None  # исходное имя времени события; None = осторожная эвристика
AS_OF_DATE = None  # пример: "2026-03-31"
WINDOWS_DAYS = (30, 90)
TABLE_SETTINGS = {}  # правила других источников, если их смысл проверен вручную
EXAMPLE_SOURCE_SHA256 = "cc644f619df766023ff1702e1b8ad222e132eff57d5c015d2af5d55a56acecd5"
EXAMPLE_TABLE_SETTINGS = {
        # Правила только для точной копии проверенного тестового набора.
        "id_column": "client_id",
        "time_column": "event_timestamp",
        "unique_key_columns": ["event_id"],
        # Эти поля уже являются скользящими показателями в отдельных строках.
        # Медиана по старым событиям не дает текущее значение за 30/90 дней.
        "exclude_source_fields": ["expense_count_prior_30d", "expense_sum_prior_30d_rub",
                                  "app_visits_prior_30d", "days_since_prior_app_visit",
                                  "campaign_touches_prior_90d", "campaign_id",
                                  "expense_amount_rub", "balance_change_rub"],
        "numeric_aggregate_preferences": {"balance_after_event_rub": "latest"},
        "behavior_rollups": [
            {"name": "expenses_total", "where": {"event_type": ["расход"]},
             "amount_column": "expense_amount_rub", "windows": [30, 90]},
            {"name": "expenses_by_account", "where": {"event_type": ["расход"]},
             "category_column": "account_scope", "category_values": ["личный счет", "счет ИП"],
             "amount_column": "expense_amount_rub", "windows": [30, 90]},
            {"name": "expenses_by_category", "where": {"event_type": ["расход"]},
             "category_column": "expense_category",
             "category_values": ["аптеки", "выплаты персоналу", "закупки продуктов для заведения",
                                 "кафе", "коммунальные услуги", "одежда", "продукты", "стриминг", "фитнес"],
             "amount_column": "expense_amount_rub", "windows": [90]},
        ],
}
NORMALIZATION_PARAMS_PATH = None  # путь к normalization_params.json обучающего периода; None = fit текущего набора
MAX_MISSING_SHARE = 0.5  # более пустые поля не входят в матрицу при первом обучении параметров
MAX_CATEGORIES = 12  # остальные значения каждой категории входят в группу «прочие»
EXCLUDE_SOURCE_FIELDS = ()  # необязательный список исходных полей, которые нельзя использовать как признаки
OUTPUT_DIR = Path(os.getenv("AUDIENCE_OUTPUT_DIR", "audience_preparation_output"))


## 2. Загрузка и чтение

В Colab откроется окно выбора исходных файлов. Листы без заголовков и пустые листы попадут в список пропусков с причиной. Если у реальной таблицы заголовки начинаются ниже первой строки, задайте `HEADER_ROWS` и перезапустите ноутбук. Исходный файл не изменяется.

Ячейки XLSX, записанные как текст, читаются без преобразования в число на входе. Так сохраняются коды и длинные идентификаторы с ведущими нулями.


In [ ]:
if not INPUT_PATHS:
    if "google.colab" not in sys.modules:
        raise ValueError("Укажите INPUT_PATHS либо загрузите файл через Google Colab")
    from google.colab import files
    uploaded = files.upload()
    INPUT_PATHS = list(uploaded.keys())

print("Выбрано файлов:", len(INPUT_PATHS))
import hashlib
for input_path in INPUT_PATHS:
    with open(input_path, "rb") as source_file:
        source_hash = hashlib.file_digest(source_file, "sha256").hexdigest()
    filename = Path(input_path).name
    if source_hash == EXAMPLE_SOURCE_SHA256:
        if filename not in TABLE_SETTINGS:
            TABLE_SETTINGS[filename] = EXAMPLE_TABLE_SETTINGS
        print(f"{filename}: применены проверенные правила тестового набора (ID, время, расходы)")
    elif filename not in TABLE_SETTINGS and not any(isinstance(key, tuple) and key[0] == filename for key in TABLE_SETTINGS):
        print(f"{filename}: бизнес-правила не заданы; матрица будет только техническим черновиком")


In [ ]:
"""Schema-agnostic, conservative preparation of tabular audience data.

No business field names are encoded. The original field names are preserved.
"""

from __future__ import annotations

import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd


SUPPORTED_TYPES = {"empty", "boolean", "datetime", "numeric", "numeric_discrete", "categorical", "identifier", "text", "mixed"}
BOOL_TRUE = {"true", "yes", "да", "истина"}
BOOL_FALSE = {"false", "no", "нет", "ложь"}
# Examples to show for review, never converted to missing without a declaration.
POSSIBLE_MISSING_MARKERS = {"null", "n/a", "н/д", "нет данных", "неизвестно", "unknown"}
DATE_SHAPE = re.compile(r"^\s*(?:\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[.\-/]\d{1,2}[.\-/]\d{4})")
AMBIGUOUS_DATE = re.compile(r"^\s*(\d{1,2})[.\-/](\d{1,2})[.\-/]\d{4}(?!\d)")
MAX_EXACT_FLOAT_INTEGER = 2**53


def _nonblank(series):
    return series.map(lambda v: pd.notna(v) and str(v).strip() != "")


def _ambiguous_day_month(value):
    match = AMBIGUOUS_DATE.match(str(value))
    return bool(match and 1 <= int(match.group(1)) <= 12 and 1 <= int(match.group(2)) <= 12)


def _as_text(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA
    if isinstance(value, (float, np.floating)) and math.isfinite(value) and value.is_integer() and abs(value) <= MAX_EXACT_FLOAT_INTEGER:
        return str(int(value))
    return str(value).strip()


def _integer_precision_risk(value):
    text = str(value).lstrip("+-")
    if not text.isdecimal():
        return False
    return len(text) > 16 or (len(text) >= 15 and int(text) > MAX_EXACT_FLOAT_INTEGER)


def _as_number(value):
    """Parse ordinary numerics; leave ambiguous grouping separators unresolved."""
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    if isinstance(value, (int, float, np.number)):
        if isinstance(value, (int, np.integer)) and abs(int(value)) > MAX_EXACT_FLOAT_INTEGER:
            return np.nan  # float conversion could silently change a large integer
        result = float(value)
        return result if math.isfinite(result) else np.nan
    text = str(value).strip().replace("\u00a0", "").replace(" ", "")
    if text.count(",") > 1 or text.count(".") > 1 or ("," in text and "." in text):
        return np.nan
    if re.fullmatch(r"[-+]?\d{1,3}[,.]\d{3}", text):
        return np.nan  # '1,000' / '1.000' can mean 1000 or 1.000
    if _integer_precision_risk(text):
        return np.nan
    try:
        result = float(text.replace(",", "."))
        return result if math.isfinite(result) else np.nan
    except ValueError:
        return np.nan


def _as_timestamp(value, source_timezone="UTC", dayfirst=False, date_format=None):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    try:
        if isinstance(value, (int, float, np.number)):
            if 20000 <= float(value) <= 100000:  # common Excel serial date range
                result = pd.to_datetime(value, origin="1899-12-30", unit="D")
            else:
                return pd.NaT  # numeric epoch units require explicit source logic
        else:
            result = pd.to_datetime(value, errors="raise", dayfirst=dayfirst, format=date_format)
        result = pd.Timestamp(result)
        if result.tzinfo is None:
            return result.tz_localize(source_timezone, ambiguous="raise", nonexistent="raise").tz_convert("UTC")
        return result.tz_convert("UTC")
    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def _csv(path, header_row=0):
    error = None
    for encoding in ("utf-8-sig", "cp1251"):
        try:
            separator = "\t" if path.suffix.lower() == ".tsv" else None
            options = {"sep": separator, "engine": "python", "dtype": str, "encoding": encoding,
                       "keep_default_na": False, "na_values": [""], "header": header_row}
            raw_header = pd.read_csv(path, **{**options, "header": None, "skiprows": header_row, "nrows": 1}).iloc[0].tolist()
            _check_header(raw_header, f"{path.name} / CSV", header_row)
            return pd.read_csv(path, **options)
        except UnicodeError as exc:
            error = exc
    raise ValueError(f"Не удалось прочитать {path}: {error}")


def _check_header(raw_header, origin, header_row):
    values = [str(v).strip() if pd.notna(v) else "" for v in raw_header]
    if not values or any(not value for value in values):
        raise ValueError(f"Нет заголовка у одного или нескольких столбцов: {origin}; задайте HEADER_ROWS (сейчас {header_row})")
    if len(values) != len(set(values)):
        raise ValueError(f"Повторяются заголовки: {origin}; имена нельзя незаметно менять")


def _header_for(header_rows, file, sheet):
    row = header_rows.get((file, sheet), header_rows.get(file, 0))
    if not isinstance(row, int) or row < 0:
        raise ValueError(f"Номер строки заголовка должен быть целым неотрицательным числом: {file} / {sheet}")
    return row


def load_tables(paths, sheets=None, header_rows=None):
    """Process each tabular sheet separately; show skipped sheets in inventory."""
    sheets = sheets or {}
    header_rows = header_rows or {}
    tables, source_info = [], []
    seen_filenames = set()
    for name in paths:
        path = Path(name)
        if not path.exists():
            raise FileNotFoundError(path)
        if path.name in seen_filenames:
            raise ValueError(f"Повторяется имя файла {path.name!r}. Для нескольких источников нужны разные имена файлов")
        seen_filenames.add(path.name)
        if path.suffix.lower() in (".xlsx", ".xlsm"):
            try:
                book = pd.ExcelFile(path)
            except (ValueError, OSError) as exc:
                source_info.append({"file": path.name, "sheet": None, "status": "skipped_unreadable_file", "hint": str(exc)})
                continue
            selected_by_user = path.name in sheets
            selected = sheets.get(path.name, book.sheet_names)
            selected = [selected] if isinstance(selected, str) else list(selected)
            if len(selected) != len(set(selected)) or set(selected) - set(book.sheet_names):
                raise ValueError(f"Неверный список листов для {path.name}: {selected}")
            candidates = []
            for tab in book.sheet_names:
                if tab not in selected:
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_by_selection"})
                    continue
                header_row = _header_for(header_rows, path.name, tab)
                raw = pd.read_excel(book, sheet_name=tab, header=None, skiprows=header_row, nrows=1,
                                    keep_default_na=False)
                if raw.empty:
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_empty"})
                    continue
                raw_header = raw.iloc[0].tolist()
                if any(pd.isna(v) or not str(v).strip() or str(v).lower().startswith("unnamed:") for v in raw_header):
                    if selected_by_user:
                        raise ValueError(f"У выбранного листа {path.name} / {tab} нет заголовков в строке {header_row + 1}; задайте HEADER_ROWS")
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_missing_header",
                                        "hint": "Укажите HEADER_ROWS[(имя_файла, имя_листа)] для листа с таблицей"})
                    continue
                try:
                    _check_header(raw_header, f"{path.name} / {tab}", header_row)
                except ValueError as exc:
                    if selected_by_user:
                        raise
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_invalid_header", "hint": str(exc)})
                    continue
                # dtype=object preserves cells stored as Excel text. With the default
                # inference pandas turns e.g. '0059' into 59 before we can inspect it.
                frame = pd.read_excel(book, sheet_name=tab, header=header_row, dtype=object,
                                      keep_default_na=False, na_values=[""])
                if any(str(field).lower().startswith("unnamed:") for field in frame.columns):
                    if selected_by_user:
                        raise ValueError(f"У выбранного листа {path.name} / {tab} есть столбцы без имен; задайте HEADER_ROWS")
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_missing_header",
                                        "hint": "Укажите HEADER_ROWS[(имя_файла, имя_листа)] для листа с таблицей"})
                    continue
                if frame.empty:
                    source_info.append({"file": path.name, "sheet": tab, "status": "skipped_empty"})
                    continue
                candidates.append((tab, frame, header_row))
        elif path.suffix.lower() in (".csv", ".tsv"):
            header_row = _header_for(header_rows, path.name, "")
            try:
                candidates = [("", _csv(path, header_row), header_row)]
            except (ValueError, pd.errors.ParserError) as exc:
                source_info.append({"file": path.name, "sheet": None, "status": "skipped_unreadable_file", "hint": str(exc)})
                continue
        else:
            raise ValueError(f"Поддерживаются XLSX, CSV и TSV: {path.name}")
        for tab, frame, header_row in candidates:
            if frame.empty:
                source_info.append({"file": path.name, "sheet": tab or None, "status": "skipped_empty"})
                continue
            if frame.columns.duplicated().any():
                raise ValueError(f"Повторяются имена столбцов: {path.name} / {tab}")
            tables.append((path.name, tab, frame, header_row))
            source_info.append({"file": path.name, "sheet": tab or None, "status": "ready",
                                "rows": len(frame), "columns": len(frame.columns), "header_row_zero_based": header_row})
    if not tables:
        raise ValueError(f"Нет таблиц для обработки. Проверьте листы и HEADER_ROWS: {source_info}")
    return tables, source_info




## 3. Определение типов по значениям

Типы: дата/время, число, дискретное число, логическое значение, категория, идентификатор, текст, смешанный или пустой столбец. Похожее на ID поле не объявляется `client_id` по имени: технический тип и роль в данных являются разными вещами. Текстовые коды с ведущим нулем, повторяющиеся цифровые значения одинаковой длины и большие целые числа не переводятся в `float`, где можно потерять представление или точность. Дискретные числа могут быть и счетчиками, и кодами: это отмечается для проверки.


In [ ]:
def infer_type(series, override=None):
    """Infer storage/validation type from values, not from the column name."""
    if override is not None:
        if override not in SUPPORTED_TYPES:
            raise ValueError(f"Неизвестный тип {override!r}; допустимо: {sorted(SUPPORTED_TYPES)}")
        return override, {"reason": "выбран вручную"}
    nonempty = series[_nonblank(series)]
    if nonempty.empty:
        return "empty", {"reason": "все значения пустые"}
    if pd.api.types.is_bool_dtype(nonempty):
        return "boolean", {"reason": "логический тип в источнике"}
    if pd.api.types.is_datetime64_any_dtype(nonempty):
        return "datetime", {"reason": "тип даты в источнике", "parse_ratio": 1.0}
    # A small vocabulary of equally wide digit strings is likely a code.
    # Preserve its exact representation, even when no value starts with zero.
    token_sample = nonempty.astype(str).str.strip().sample(n=min(len(nonempty), 2000), random_state=42)
    if (len(nonempty) >= 30 and token_sample.str.fullmatch(r"\d{3,8}").all()
            and token_sample.str.len().nunique() == 1 and token_sample.nunique() <= 20):
        return "categorical", {"reason": "равная длина повторяющихся цифровых значений: возможный код; смысл проверьте",
                               "unique_values": int(nonempty.nunique())}
    if pd.api.types.is_numeric_dtype(nonempty):
        if not pd.api.types.is_complex_dtype(nonempty):
            large_integers = nonempty.map(lambda v: isinstance(v, (int, np.integer)) and abs(int(v)) > MAX_EXACT_FLOAT_INTEGER)
            if large_integers.mean() >= 0.8:
                return "identifier", {"reason": "целые значения слишком велики для точного перевода в float", "review": "проверьте роль поля"}
            unique = nonempty.nunique()
            discrete = unique <= 20 and (nonempty.dropna() % 1 == 0).all()
            return ("numeric_discrete" if discrete else "numeric"), {"reason": "числовой тип в источнике", "unique_values": int(unique)}
    values = nonempty.astype(str).str.strip()
    sample = values.sample(n=min(len(values), 2000), random_state=42)
    lowered = set(sample.str.casefold().unique())
    if lowered and lowered.issubset(BOOL_TRUE | BOOL_FALSE):
        return "boolean", {"reason": "явные логические значения", "parse_ratio": 1.0}
    date_candidate = sample.map(lambda s: bool(DATE_SHAPE.match(s)))
    if date_candidate.mean() >= 0.8:
        parsed = sample.map(_as_timestamp)
        ratio = float(parsed.notna().mean())
        if ratio >= 0.8:
            return "datetime", {"reason": "значения похожи на даты", "parse_ratio": round(ratio, 3)}
    leading_zero_code = sample.map(lambda s: bool(re.fullmatch(r"0\d+", s))).any()
    large_integer_code = sample.map(_integer_precision_risk).mean()
    if large_integer_code >= 0.8:
        return "identifier", {"reason": "длинные целые значения: риск потери точности при числовом преобразовании"}
    numeric = sample.map(_as_number)
    ratio = float(numeric.notna().mean())
    if ratio >= 0.8 and not leading_zero_code:
        distinct = nonempty.nunique()
        discrete = distinct <= 20 and numeric.dropna().mod(1).eq(0).all()
        return ("numeric_discrete" if discrete else "numeric"), {"reason": "значения похожи на числа", "parse_ratio": round(ratio, 3)}
    distinct = nonempty.astype(str).str.strip().nunique()
    short_tokens = sample.map(lambda s: len(s) <= 64 and len(s.split()) <= 3).mean()
    if 0.2 <= ratio < 0.8:
        return "mixed", {"reason": "смешанные числовые и нечисловые значения", "numeric_parse_ratio": round(ratio, 3)}
    if short_tokens >= 0.85 and (
        distinct > max(50, math.sqrt(len(nonempty)))
        or (distinct >= 5 and distinct / len(nonempty) >= 0.8)
    ):
        return "identifier", {"reason": "много различных коротких значений", "unique_values": int(distinct)}
    if distinct <= max(20, math.sqrt(len(nonempty))) and distinct / len(nonempty) <= 0.5:
        return "categorical", {"reason": "повторяющиеся значения", "unique_values": int(distinct)}
    return "text", {"reason": "строковые значения высокой вариативности", "unique_values": int(distinct)}




## 4. Очистка и контроль качества по типу

Считаются пропуски и ошибки разбора; для непрерывных чисел отмечаются отрицательные значения и большие значения относительно IQR, для дат — диапазон и возможная неоднозначность формата, для категорий — редкие и различающиеся регистром варианты. Возможные маркеры пропусков перечисляются для подтверждения. Для дискретных чисел выбросы не ищутся: это могут быть коды. Идентификаторы проверяются на повторяемость, текст — на длину. Сам по себе отрицательный баланс или пустой MCC не считается ошибкой без знания смысла поля.

Исходные названия колонок сохраняются. Существующие пропуски не заполняются. Если в колонке встретились ошибки преобразования, ее исходные значения сохраняются в `prepared_XX.csv`, а номера проблемных строк указываются отдельно. В аналитической копии отдельные неверные значения числового/логического поля становятся пустыми, при этом все корректные значения преобразуются. Неоднозначная дата остается без преобразования до задания формата. Категории, различающиеся только регистром, унифицируются; предположения о синонимах и маркерах пропуска не делаются.


In [ ]:
def _field_quality(original, prepared, inferred_type, metadata, errors, review_flags=None):
    nonempty = original[_nonblank(original)]
    nonempty_clean = prepared[_nonblank(prepared)]
    result = {
        "field": str(original.name),
        "source_dtype": str(original.dtype),
        "inferred_type": inferred_type,
        "rows": int(len(original)),
        "missing_count": int(len(original) - len(nonempty)),
        "missing_share": round(1 - len(nonempty) / len(original), 4),
        "distinct_nonempty": int(nonempty_clean.nunique(dropna=True)),
        "parse_failures": int(errors),
        "conversion_applied": bool(inferred_type in ("numeric", "numeric_discrete", "datetime", "boolean")
                                   and errors == 0 and not any("skipped" in flag or "ambiguous_day_month" in flag
                                                               for flag in (review_flags or []))),
        "inference": metadata,
        "review_flags": review_flags or [],
    }
    if inferred_type == "numeric":
        nums = prepared.map(_as_number).dropna()
        if len(nums):
            q1, q3 = nums.quantile([0.25, 0.75])
            result.update({"min": float(nums.min()), "median": float(nums.median()),
                           "p99": float(nums.quantile(0.99)), "max": float(nums.max())})
            result["negative_count_flag"] = int((nums < 0).sum())
            result["high_iqr_outliers_flag"] = int((nums > q3 + 1.5 * (q3 - q1)).sum())
    elif inferred_type == "datetime":
        dates = pd.to_datetime(prepared, utc=True, errors="coerce").dropna() if pd.api.types.is_datetime64_any_dtype(prepared) else pd.Series(dtype="datetime64[ns, UTC]")
        if len(dates):
            result.update({"earliest_utc": str(dates.min()), "latest_utc": str(dates.max())})
        result["ambiguous_day_month_format_flag"] = int(nonempty.map(_ambiguous_day_month).sum())
    elif inferred_type == "categorical":
        freq = nonempty_clean.astype(str).value_counts()
        result["rare_values_under_3_rows"] = int(freq.lt(3).sum())
        result["case_variant_groups_flag"] = int(len(freq) - len({k.casefold() for k in freq.index}))
    elif inferred_type == "identifier":
        result["repeated_nonempty_rows"] = int(nonempty_clean.duplicated(keep=False).sum())
        result["unique_share_nonempty"] = round(nonempty_clean.nunique() / len(nonempty_clean), 4) if len(nonempty_clean) else None
    elif inferred_type == "text":
        result["max_length"] = int(nonempty_clean.astype(str).str.len().max()) if len(nonempty_clean) else 0
    return result


def prepare_table(frame, *, type_overrides=None, timezone="UTC", dayfirst=None, date_formats=None,
                  missing_tokens=(), header_row=0):
    """Preserve column names; convert only confidently inferred types."""
    type_overrides = type_overrides or {}
    date_formats = date_formats or {}
    prepared = frame.copy()
    field_report = []
    row_issues = []
    schema = {}
    missing_tokens = {str(token).strip().casefold() for token in missing_tokens}
    for field in frame.columns:
        original = frame[field].copy()
        if missing_tokens:
            declared_missing = original.map(lambda v: isinstance(v, str) and v.strip().casefold() in missing_tokens)
            for position in frame.index[declared_missing]:
                row_issues.append({"source_row": int(position) + header_row + 2, "field": str(field),
                                   "issue": "declared_missing_marker_normalized"})
            original.loc[declared_missing] = pd.NA
        prepared[field] = original
        kind, evidence = infer_type(original, type_overrides.get(field))
        schema[field] = kind
        nonblank = _nonblank(original)
        failures = pd.Series(False, index=frame.index)
        review_flags = []
        candidate_markers = original[nonblank].map(str).str.strip()
        candidate_markers = candidate_markers[
            candidate_markers.str.casefold().isin(POSSIBLE_MISSING_MARKERS)
        ]
        if not candidate_markers.empty:
            review_flags.append("possible_missing_markers; confirm before adding to MISSING_TOKENS")
        if original.map(lambda v: isinstance(v, (float, np.floating)) and pd.notna(v)
                        and abs(v) > MAX_EXACT_FLOAT_INTEGER).any():
            review_flags.append("large_float_in_source; identifier precision may already be lost")
        if kind in ("numeric", "numeric_discrete"):
            parsed = original.map(_as_number)
            failures = nonblank & parsed.isna()
            if not failures.any():
                prepared[field] = parsed
            else:
                review_flags.append("numeric_conversion_skipped_due_to_parse_failures")
            if kind == "numeric_discrete":
                review_flags.append("may_be_numeric_code_or_count; no_outlier_detection")
        elif kind == "datetime":
            fmt = date_formats.get(field)
            if fmt is None and dayfirst is None and original[nonblank].map(_ambiguous_day_month).any():
                review_flags.append("ambiguous_day_month; set DAYFIRST or DATE_FORMATS to convert")
            else:
                parsed = original.map(lambda v: _as_timestamp(v, timezone, bool(dayfirst), fmt))
                failures = nonblank & parsed.isna()
                if not failures.any():
                    prepared[field] = pd.to_datetime(parsed, utc=True)
                else:
                    review_flags.append("datetime_conversion_skipped_due_to_parse_failures")
        elif kind == "boolean":
            def cast_bool(v):
                if pd.isna(v) or str(v).strip() == "":
                    return pd.NA
                if isinstance(v, (bool, np.bool_)):
                    return bool(v)
                key = str(v).strip().casefold()
                if key in BOOL_TRUE:
                    return True
                if key in BOOL_FALSE:
                    return False
                return pd.NA
            parsed = original.map(cast_bool)
            failures = nonblank & parsed.isna()
            if not failures.any():
                prepared[field] = parsed.astype("boolean")
            else:
                review_flags.append("boolean_conversion_skipped_due_to_parse_failures")
        elif kind in ("categorical", "identifier", "text", "mixed"):
            prepared[field] = original.map(_as_text).astype("string")
            if kind == "mixed":
                review_flags.append("mixed_values; choose TYPE_OVERRIDES only after checking the source")
        # For 'empty', keep the original empty column and its name.
        for position in frame.index[failures]:
            row_issues.append({"source_row": int(position) + header_row + 2, "field": str(field), "issue": "value_could_not_be_parsed"})
        field_quality = _field_quality(original, prepared[field], kind, evidence, int(failures.sum()), review_flags)
        field_quality["possible_missing_markers"] = candidate_markers.value_counts().to_dict()
        field_report.append(field_quality)
    exact_duplicate = prepared.duplicated(keep=False)
    for position in frame.index[exact_duplicate]:
        row_issues.append({"source_row": int(position) + header_row + 2, "field": None, "issue": "exact_duplicate_row_flagged_not_removed"})
    return prepared, pd.DataFrame(field_report), pd.DataFrame(row_issues, columns=["source_row", "field", "issue"]), schema


def build_analysis_view(prepared, schema, *, timezone="UTC", dayfirst=None, date_formats=None):
    """Typed derivative: invalid cells are null only here; prepared remains auditable."""
    result = prepared.copy()
    conversions = []
    date_formats = date_formats or {}
    for field, kind in schema.items():
        values = prepared[field]
        nonblank = _nonblank(values)
        invalid = 0
        normalized = 0
        status = "unchanged"
        if kind in ("numeric", "numeric_discrete"):
            parsed = values.map(_as_number)
            invalid = int((nonblank & parsed.isna()).sum())
            result[field] = pd.to_numeric(parsed, errors="coerce")
            status = "parsed_with_invalid_as_missing" if invalid else "parsed"
        elif kind == "boolean":
            def to_bool(value):
                if pd.isna(value) or str(value).strip() == "":
                    return pd.NA
                if isinstance(value, (bool, np.bool_)):
                    return bool(value)
                text = str(value).strip().casefold()
                if text in BOOL_TRUE:
                    return True
                if text in BOOL_FALSE:
                    return False
                return pd.NA
            parsed = values.map(to_bool)
            invalid = int((nonblank & parsed.isna()).sum())
            result[field] = parsed.astype("boolean")
            status = "parsed_with_invalid_as_missing" if invalid else "parsed"
        elif kind == "datetime":
            fmt = date_formats.get(field)
            if fmt is None and dayfirst is None and values[nonblank].map(_ambiguous_day_month).any():
                status = "blocked_ambiguous_date"
            else:
                parsed = values.map(lambda value: _as_timestamp(value, timezone, bool(dayfirst), fmt))
                invalid = int((nonblank & parsed.isna()).sum())
                result[field] = pd.to_datetime(parsed, utc=True)
                status = "parsed_with_invalid_as_missing" if invalid else "parsed"
        elif kind == "categorical":
            # Only casing is standardized; different spellings or unknown markers
            # need a source-specific dictionary, not an inferred equivalence.
            text_values = values.map(_as_text).astype("string")
            frequency = text_values.dropna().value_counts()
            canonical = {}
            for spelling, count in frequency.items():
                folded = spelling.casefold()
                if folded not in canonical or count > canonical[folded][1]:
                    canonical[folded] = (spelling, count)
            replacements = {spelling: canonical[spelling.casefold()][0] for spelling in frequency.index}
            standardized = text_values.map(lambda value: replacements.get(value, value) if pd.notna(value) else pd.NA).astype("string")
            normalized = int((text_values.notna() & text_values.ne(standardized).fillna(False)).sum())
            result[field] = standardized
            status = "case_variants_standardized" if normalized else "unchanged"
        conversions.append({"field": str(field), "status": status, "invalid_as_missing": invalid,
                            "case_normalized_cells": normalized})
    return result, conversions


def check_unique_record_key(prepared, unique_key_columns=None, header_row=0, schema=None):
    """Check declared grain or flag suspicious nearly unique IDs without assuming their role."""
    if unique_key_columns is None:
        schema = schema or {}
        time_fields = [field for field, kind in schema.items() if kind == "datetime"]
        candidates, suspected, uncertain, exact = [], set(), set(), set()
        issues = []
        for field, kind in schema.items():
            if kind != "identifier":
                continue
            values = prepared[field].dropna()
            if len(values) < 10 or values.nunique() / len(values) < 0.9:
                continue
            candidates.append(field)
            repeated = prepared[prepared[field].notna() & prepared[field].duplicated(keep=False)]
            for _, group in repeated.groupby(field, sort=False, dropna=False):
                if group.drop_duplicates().shape[0] == 1:
                    exact.update(group.index.tolist())
                    continue
                # The same identifier at the same observed time with different
                # payload is suspicious. Across dates it can be legitimate history.
                same_observed_time = any(
                    group[time_field].notna().all() and group[time_field].nunique(dropna=False) == 1
                    for time_field in time_fields
                )
                target = suspected if same_observed_time else uncertain
                target.update(group.index.tolist())
                issue = "possible_record_id_conflict_same_time" if same_observed_time else "repeated_identifier_needs_review"
                for position in group.index:
                    issues.append({"source_row": int(position) + header_row + 2,
                                   "field": str(field), "issue": issue})
        return pd.DataFrame(issues, columns=["source_row", "field", "issue"]), {
            "status": "auto_candidates_need_review" if candidates else "no_confident_candidate",
            "columns": [], "candidate_columns": candidates, "conflict_rows": 0,
            "suspected_conflict_rows": len(suspected),
            "uncertain_repeat_rows": len(uncertain - suspected),
            "exact_repeat_rows": len(exact), "missing_key_rows": 0,
            "warning": "Кандидаты найдены по значениям; уникальность записи и смысл времени требуют проверки",
        }
    if isinstance(unique_key_columns, str) or not isinstance(unique_key_columns, (list, tuple)) or not unique_key_columns:
        raise ValueError("unique_key_columns: задайте непустой список исходных имен полей")
    key_columns = list(unique_key_columns)
    if len(set(key_columns)) != len(key_columns) or any(k not in prepared.columns for k in key_columns):
        raise ValueError(f"unique_key_columns: повторяющиеся или отсутствующие поля: {key_columns}")
    missing_key = prepared[key_columns].isna().any(axis=1)
    complete = prepared.loc[~missing_key]
    repeated = complete.duplicated(subset=key_columns, keep=False)
    conflict_rows = []
    exact_repeat_rows = []
    for _, group in complete.loc[repeated].groupby(key_columns, dropna=False, sort=False):
        if group.duplicated(keep=False).all() and group.drop_duplicates().shape[0] == 1:
            exact_repeat_rows.extend(group.index.tolist())
        else:
            conflict_rows.extend(group.index.tolist())
    field = ", ".join(map(str, key_columns))
    issues = []
    for indices, issue in (
        (prepared.index[missing_key], "missing_declared_record_key"),
        (exact_repeat_rows, "duplicate_record_key_same_values"),
        (conflict_rows, "duplicate_record_key_conflicting_values"),
    ):
        for position in indices:
            issues.append({"source_row": int(position) + header_row + 2, "field": field, "issue": issue})
    return pd.DataFrame(issues, columns=["source_row", "field", "issue"]), {
        "status": "checked", "columns": key_columns,
        "conflict_rows": len(conflict_rows),
        "suspected_conflict_rows": 0, "uncertain_repeat_rows": 0,
        "exact_repeat_rows": len(exact_repeat_rows),
        "missing_key_rows": int(missing_key.sum()),
    }




## 5. История и предварительные признаки на клиента

`client_index_XX.csv` сохраняет прежнюю строгую проверку: при подозрительном конфликте ID записи он не создается. Новый `client_features_XX.csv` делает предварительную сборку одной строки на ID. Проблемные пары строк с конфликтующим ID и точные копии исключаются только из расчета признаков, их число записывается в отчете. Для дат считаются первая/последняя даты и частота за 30/90 дней; для чисел — медиана и диапазон; для категорий — разнообразие и частое значение. Сумма разрешена только через `sum_columns` или явные `behavior_rollups`. Последние считают число событий и сумму указанного числового поля только по выбранным условиям, с разбивкой по значениям категории и временным окнам. Если в статичной таблице ID встречается с разными значениями и нет надежного времени, признаки по этой таблице не создаются.

`client_features_combined.csv` связывает таблицы только если имя ID совпадает и значения ID имеют большое пересечение. Исходные поля не переименовываются: префиксы `__tXX__` применяются только к вычисленным признакам. Совпадение по значениям не доказывает семантическую тождественность ID. При заданном `AS_OF_DATE` таблица без разобранного времени не участвует в историческом срезе. До ML проверьте исключенные записи, смысл чисел, допустимость категорий и реальную дату среза.


In [ ]:
def infer_history_roles(prepared, schema, id_column=None, time_column=None):
    """Choose only when values make roles clear; allow explicit user overrides."""
    if id_column is not None and id_column not in prepared:
        raise ValueError(f"ID_COLUMN {id_column!r} отсутствует")
    if time_column is not None and time_column not in prepared:
        raise ValueError(f"TIME_COLUMN {time_column!r} отсутствует")
    if id_column is None:
        candidates = []
        for field, kind in schema.items():
            if kind != "identifier":
                continue
            unique = prepared[field].nunique(dropna=True)
            populated = prepared[field].notna().sum()
            if populated >= 2 and 1 < unique and unique / populated < 0.85:
                candidates.append((unique / populated, field))
        if len(candidates) == 1:
            id_column = candidates[0][1]
    if time_column is None:
        candidates = []
        for field, kind in schema.items():
            if kind == "datetime" and pd.api.types.is_datetime64_any_dtype(prepared[field]):
                populated = prepared[field].notna().sum()
                distinct = prepared[field].nunique(dropna=True)
                if populated:
                    candidates.append((distinct / populated, field))
        candidates.sort(reverse=True)
        if candidates and (len(candidates) == 1 or candidates[0][0] > 1.5 * candidates[1][0]):
            time_column = candidates[0][1]
    return id_column, time_column


def _as_of(value, dates):
    if value is None or str(value).strip() == "":
        return dates.max()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(value)):
        return pd.Timestamp(value, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
    ts = _as_timestamp(value)
    if pd.isna(ts):
        raise ValueError("AS_OF_DATE: задайте дату YYYY-MM-DD или ISO timestamp")
    return ts


def build_client_index(prepared, id_column, time_column, as_of_date=None, windows=(30, 90),
                       record_key_report=None):
    """Minimal client-time index. No business sums or invented ML features."""
    if not id_column or not time_column:
        return None, {"status": "skipped", "reason": "Неоднозначны ID и/или время события: заполните ID_COLUMN и TIME_COLUMN"}
    if record_key_report and record_key_report["conflict_rows"]:
        return None, {"status": "skipped", "reason": "Повторяющийся ключ записи с разным содержимым: проверьте row_issues перед подсчетом",
                      "conflicting_record_key_rows": record_key_report["conflict_rows"]}
    if record_key_report and record_key_report.get("suspected_conflict_rows", 0):
        return None, {"status": "skipped", "reason": "Обнаружены возможные конфликты повторяющегося ID в одно время: проверьте row_issues и смысл ключа записи",
                      "suspected_conflict_rows": record_key_report["suspected_conflict_rows"]}
    if not pd.api.types.is_datetime64_any_dtype(prepared[time_column]):
        return None, {"status": "skipped", "reason": "Время не преобразовано в дату: проверьте качество и DATE_FORMATS"}
    # Keep raw/prepared rows intact. The derived count must not include an
    # identical imported copy twice. Other duplicate keys require review.
    duplicated_copy = prepared.duplicated(keep="first")
    indexed = prepared.loc[~duplicated_copy]
    dates = pd.to_datetime(indexed[time_column], utc=True, errors="coerce")
    if dates.isna().all():
        return None, {"status": "skipped", "reason": "Нет корректных дат событий"}
    as_of = _as_of(as_of_date, dates.dropna())
    valid = indexed[id_column].notna() & dates.notna() & dates.le(as_of)
    d = indexed.loc[valid, [id_column]].copy()
    d["_observed_at_utc"] = dates.loc[valid]
    if d.empty:
        return None, {"status": "skipped", "reason": "Нет событий до даты среза", "as_of_utc": str(as_of)}
    g = d.groupby(id_column, dropna=True)["_observed_at_utc"]
    result = pd.DataFrame({"__observed_event_count_all": g.size(), "__first_observed_at_utc": g.min(), "__last_observed_at_utc": g.max()})
    for days in windows:
        if days <= 0:
            raise ValueError("Окна должны быть положительными")
        window = d[d["_observed_at_utc"].gt(as_of - pd.Timedelta(days=days))]
        result[f"__observed_event_count_{days}d"] = window.groupby(id_column).size().reindex(result.index, fill_value=0)
    result.reset_index(inplace=True)
    return result, {
        "status": "ready", "id_column": id_column, "time_column": time_column, "as_of_utc": str(as_of),
        "clients": len(result), "rows_missing_id_or_time": int((indexed[id_column].isna() | dates.isna()).sum()),
        "future_rows_excluded": int(dates.gt(as_of).sum()),
        "exact_duplicate_rows_excluded_from_index": int(duplicated_copy.sum()),
        "record_key_check": (
            "declared_key_checked" if record_key_report and record_key_report["status"] == "checked"
            else "auto_candidates_need_review; uniqueness of a record not established"
            if record_key_report and record_key_report["status"] == "auto_candidates_need_review"
            else "no_confident_candidate; record grain unknown"
        ),
        "observation_warning": "Ноль означает отсутствие наблюдаемых строк в загруженном источнике, а не доказанное отсутствие активности",
    }


def build_client_features(analysis, schema, issues, *, table_number, header_row, id_column,
                          time_column=None, as_of_date=None, windows=(30, 90), sum_columns=(),
                          allow_timeless_features=False, behavior_rollups=()):
    """Build a cautious per-entity preview, excluding disputed records in this derivative."""
    if id_column is None:
        return None, {"status": "skipped", "reason": "Не определен ID сущности; укажите id_column для этой таблицы"}
    if id_column not in analysis:
        raise ValueError(f"id_column {id_column!r} отсутствует")
    sum_columns = tuple(sum_columns)
    if any(column not in analysis or schema.get(column) != "numeric" for column in sum_columns):
        raise ValueError("sum_columns: укажите только существующие непрерывные числовые поля")
    if time_column is not None and time_column not in analysis:
        raise ValueError(f"time_column {time_column!r} отсутствует")
    conflict_codes = {"possible_record_id_conflict_same_time", "duplicate_record_key_conflicting_values",
                      "missing_declared_record_key"}
    conflict_rows = set((issues.loc[issues["issue"].isin(conflict_codes), "source_row"].astype(int)
                         - header_row - 2).tolist())
    conflict_rows = analysis.index.intersection(list(conflict_rows))
    duplicates = analysis.duplicated(keep="first")
    usable = analysis.loc[~analysis.index.isin(conflict_rows) & ~duplicates].copy()
    usable["__entity_id"] = analysis.loc[usable.index, id_column].map(_as_text).astype("string")
    missing_id = int(usable["__entity_id"].isna().sum())
    usable = usable.loc[usable["__entity_id"].notna()].copy()

    time_parsed = time_column is not None and pd.api.types.is_datetime64_any_dtype(analysis[time_column])
    missing_time = future_rows = 0
    as_of = None
    if as_of_date is not None and not time_parsed and not allow_timeless_features:
        return None, {"status": "skipped", "reason": "Для исторического среза нужно определить и разобрать время таблицы"}
    if time_parsed:
        timestamps = pd.to_datetime(usable[time_column], utc=True, errors="coerce")
        missing_time = int(timestamps.isna().sum())
        usable = usable.loc[timestamps.notna()].copy()
        timestamps = timestamps.dropna()
        if timestamps.empty:
            return None, {"status": "skipped", "reason": "Нет записей с корректными ID и временем"}
        as_of = _as_of(as_of_date, timestamps)
        future_rows = int(timestamps.gt(as_of).sum())
        usable = usable.loc[timestamps.le(as_of)].copy()
        usable["__observed_at_utc"] = timestamps.loc[usable.index]
        usable.sort_values("__observed_at_utc", kind="stable", inplace=True)
    elif usable["__entity_id"].duplicated().any():
        return None, {"status": "skipped", "reason": "У ID несколько разных строк и нет надежного времени для выбора или истории"}
    if usable.empty:
        return None, {"status": "skipped", "reason": "После проверки не осталось подходящих записей"}

    prefix = f"__t{table_number:02d}__"
    groups = usable.groupby("__entity_id", sort=True, dropna=True)
    result = groups.size().rename(prefix + "observed_row_count").to_frame()
    if time_parsed:
        result[prefix + "first_observed_at_utc"] = groups["__observed_at_utc"].min()
        result[prefix + "last_observed_at_utc"] = groups["__observed_at_utc"].max()
        for days in windows:
            if not isinstance(days, int) or days <= 0:
                raise ValueError("WINDOWS_DAYS: окна должны быть положительными целыми числами")
            observed = usable[usable["__observed_at_utc"].gt(as_of - pd.Timedelta(days=days))]
            result[prefix + f"observed_row_count_{days}d"] = (
                observed.groupby("__entity_id").size().reindex(result.index, fill_value=0)
            )
    for field, kind in schema.items():
        if field in (id_column, time_column) or kind in ("identifier", "empty", "mixed", "datetime"):
            continue
        col = prefix + str(field) + "__"
        if kind == "numeric":
            series = pd.to_numeric(usable[field], errors="coerce")
            numeric = series.groupby(usable["__entity_id"], sort=True)
            result[col + "valid_count"] = numeric.count()
            result[col + "median"] = numeric.median()
            result[col + "min"] = numeric.min()
            result[col + "max"] = numeric.max()
            if field in sum_columns:
                result[col + "sum"] = numeric.sum(min_count=1)
            if time_parsed or not usable["__entity_id"].duplicated().any():
                result[col + "latest"] = numeric.last()
        elif kind == "numeric_discrete":
            values = usable[field].groupby(usable["__entity_id"], sort=True)
            result[col + "distinct_count"] = values.nunique(dropna=True)
            result[col + "latest"] = values.last()
        elif kind in ("categorical", "boolean"):
            values = usable[field].groupby(usable["__entity_id"], sort=True)
            result[col + "distinct_count"] = values.nunique(dropna=True)
            if kind == "categorical":
                result[col + "most_common"] = values.agg(
                    lambda s: sorted(s.dropna().astype(str).value_counts().items(),
                                     key=lambda item: (-item[1], item[0]))[0][0] if s.notna().any() else pd.NA
                )
            if time_parsed or not usable["__entity_id"].duplicated().any():
                result[col + "latest"] = values.last()
        elif kind == "text":
            result[col + "nonmissing_count"] = usable[field].groupby(usable["__entity_id"], sort=True).count()
    # Semantics of monetary flows cannot be inferred from a numeric dtype. A
    # declared rollup counts only matching events and sums an explicitly named
    # measure, optionally broken down by fixed source values.
    rollup_fields = []
    rollup_details = []
    rollup_values = {}
    seen_names = set()
    for rule in behavior_rollups:
        if not isinstance(rule, dict) or not isinstance(rule.get("name"), str):
            raise ValueError("behavior_rollups: правило должно иметь строковое имя")
        name = rule["name"]
        if not re.fullmatch(r"[a-zA-Z0-9_]+", name) or name in seen_names:
            raise ValueError("behavior_rollups: нужны уникальные имена из букв, цифр и _")
        seen_names.add(name)
        where = rule.get("where", {})
        if not isinstance(where, dict) or any(
            field not in analysis or not isinstance(values, (tuple, list)) or not values
            for field, values in where.items()
        ):
            raise ValueError(f"behavior_rollups[{name}]: неверное условие where")
        category = rule.get("category_column")
        levels = rule.get("category_values")
        if (category is None) != (levels is None):
            raise ValueError(f"behavior_rollups[{name}]: укажите вместе category_column и category_values")
        if category is not None and (category not in analysis or not isinstance(levels, (tuple, list))
                                     or not levels or len(levels) != len(set(map(str, levels)))):
            raise ValueError(f"behavior_rollups[{name}]: неверные категории")
        amount_column = rule.get("amount_column")
        if amount_column is not None and (amount_column not in analysis or schema.get(amount_column) != "numeric"):
            raise ValueError(f"behavior_rollups[{name}]: amount_column должен быть непрерывным числом")
        rule_windows = rule.get("windows", windows)
        if (not isinstance(rule_windows, (tuple, list)) or any(not isinstance(days, int) or days <= 0
                                                              for days in rule_windows)):
            raise ValueError(f"behavior_rollups[{name}]: нужны положительные окна windows")
        if time_parsed is False and rule_windows:
            raise ValueError(f"behavior_rollups[{name}]: для окон времени нужна надежная дата")
        matching = pd.Series(True, index=usable.index)
        for field, values in where.items():
            matching &= usable[field].isin(values)
        chosen = usable.loc[matching]
        slices = [("all", chosen)]
        if time_parsed:
            slices.extend((f"{days}d", chosen.loc[chosen["__observed_at_utc"].gt(as_of - pd.Timedelta(days=days))])
                          for days in rule_windows)
        level_pairs = [("", None)] if category is None else [
            (f"value_{i:02d}", value) for i, value in enumerate(levels, start=1)
        ]
        for label, level in level_pairs:
            for period, current in slices:
                selected = current if level is None else current.loc[current[category].eq(level)]
                base = prefix + f"rollup__{name}__{label + '__' if label else ''}{period}__"
                count_field = base + "count"
                counts = selected.groupby("__entity_id").size().reindex(result.index, fill_value=0)
                rollup_values[count_field] = counts
                rollup_fields.append(count_field)
                if amount_column is not None:
                    amounts = pd.to_numeric(selected[amount_column], errors="coerce")
                    amount_field = base + "amount_sum"
                    # No matching event means zero; matching events with no
                    # valid measure retain NA so missingness is visible.
                    sums = amounts.groupby(selected["__entity_id"]).sum(min_count=1)
                    rollup_values[amount_field] = sums.reindex(result.index).where(counts.gt(0), 0)
                    rollup_fields.append(amount_field)
        rollup_details.append({"name": name, "where": where, "category_column": category,
                               "category_values": list(levels) if levels is not None else None,
                               "amount_column": amount_column, "windows": list(rule_windows),
                               "feature_columns": len(level_pairs) * len(slices) * (2 if amount_column else 1)})
    if rollup_values:
        result = pd.concat([result, pd.DataFrame(rollup_values, index=result.index)], axis=1)
    result.reset_index(inplace=True)
    return result, {
        "status": "preview_needs_role_and_quality_review",
        "entity_id_column": id_column, "time_column": time_column if time_parsed else None,
        "time_status": "parsed" if time_parsed else ("declared_timeless" if allow_timeless_features else "unavailable_or_ambiguous"),
        "allow_timeless_features": bool(allow_timeless_features),
        "as_of_utc": str(as_of) if as_of is not None else None,
        "entities": len(result), "source_rows_in_features": len(usable),
        "conflicting_rows_excluded": len(conflict_rows),
        "exact_duplicate_rows_excluded": int(duplicates.sum()),
        "missing_id_rows_excluded": missing_id, "missing_time_rows_excluded": missing_time,
        "future_rows_excluded": future_rows,
        "sum_columns_declared": list(sum_columns),
        "behavior_rollups_declared": rollup_details,
        "behavior_numeric_features": rollup_fields,
        "warning": "Проверьте роль ID, смысл числовых полей и исключенные записи перед обучением ML",
    }


def combine_client_features(results):
    """Join previews only on matching ID field names with strong value overlap."""
    candidates = [r for r in results if r.get("client_features") is not None]
    if not candidates:
        return None, {"status": "skipped", "reason": "Не построены таблицы признаков"}
    base = max(candidates, key=lambda r: (len(r["client_features"]), -r["table_number"]))
    combined = base["client_features"].copy()
    linked = [base["table_number"]]
    unlinked = []
    base_ids = set(combined["__entity_id"].dropna())
    base_field = base["features_report"]["entity_id_column"]
    for other in candidates:
        if other is base:
            continue
        field = other["features_report"]["entity_id_column"]
        other_ids = set(other["client_features"]["__entity_id"].dropna())
        overlap = len(base_ids & other_ids) / max(1, min(len(base_ids), len(other_ids)))
        if field != base_field or overlap < 0.8:
            unlinked.append({"table_number": other["table_number"], "reason": "ID не совпадает по роли/значениям"})
            continue
        combined = combined.merge(other["client_features"], on="__entity_id", how="left", validate="one_to_one")
        linked.append(other["table_number"])
    return combined, {"status": "preview_needs_role_and_quality_review", "base_table": base["table_number"],
                      "linked_tables": linked, "unlinked_tables": unlinked, "entities": len(combined),
                      "warning": "Совпадение ID проверено технически; смысл ключей и даты должны быть подтверждены до ML"}


def scale_client_numeric_features(features, fitted_params=None):
    """Robust numeric scaling after per-client aggregation; fit once, reuse on later periods."""
    if features is None:
        return None, None, {"status": "skipped", "reason": "Нет общей таблицы признаков"}, None
    if "__entity_id" not in features or features["__entity_id"].isna().any() or features["__entity_id"].duplicated().any():
        raise ValueError("Для нормализации нужны непустые уникальные ID")
    numeric_columns = [field for field in features.columns if field != "__entity_id"
                       and pd.api.types.is_numeric_dtype(features[field])
                       and not pd.api.types.is_bool_dtype(features[field])]
    mode = "fit_current_input" if fitted_params is None else "apply_saved_params"
    excluded = []
    if fitted_params is None:
        fields = []
        for field in numeric_columns:
            values = pd.to_numeric(features[field], errors="coerce").replace([np.inf, -np.inf], np.nan)
            valid = values.dropna()
            if valid.nunique() < 2:
                excluded.append({"field": str(field), "reason": "пустой или постоянный числовой признак"})
                continue
            lower, upper = float(valid.quantile(0.01)), float(valid.quantile(0.99))
            limited = valid.clip(lower=lower, upper=upper)
            median = float(limited.median())
            iqr = float(limited.quantile(0.75) - limited.quantile(0.25))
            if iqr > 0:
                scale, scale_method = iqr, "iqr"
            else:
                # Sparse counts can have zero IQR but still distinguish clients.
                scale = max(upper - lower, 1.0)
                scale_method = "p99_minus_p01_floor_1"
            fields.append({"field": str(field), "median": median, "scale": scale,
                           "lower": lower, "upper": upper,
                           "scale_method": scale_method})
        params = {"version": 1, "method": "median_iqr", "fields": fields}
    else:
        if (not isinstance(fitted_params, dict) or fitted_params.get("version") != 1
                or fitted_params.get("method") != "median_iqr" or not isinstance(fitted_params.get("fields"), list)):
            raise ValueError("SCALER_PARAMS_PATH: ожидается JSON с version=1 и method=median_iqr")
        params = fitted_params
        fields = params["fields"]
        names = [entry.get("field") for entry in fields if isinstance(entry, dict)]
        if len(names) != len(fields) or len(names) != len(set(names)) or any(name not in numeric_columns for name in names):
            raise ValueError("В данных отсутствует числовой признак из параметров нормализации или он изменил тип")
        for entry in fields:
            if (not isinstance(entry.get("median"), (int, float)) or not math.isfinite(entry["median"])
                    or not isinstance(entry.get("scale"), (int, float)) or not math.isfinite(entry["scale"])
                    or entry["scale"] <= 0 or not isinstance(entry.get("lower"), (int, float))
                    or not isinstance(entry.get("upper"), (int, float))
                    or not math.isfinite(entry["lower"]) or not math.isfinite(entry["upper"])
                    or entry["lower"] > entry["upper"]):
                raise ValueError("Некорректные границы/median/scale в параметрах нормализации")
        excluded = [{"field": str(field), "reason": "нет в базовых параметрах"}
                    for field in numeric_columns if field not in set(names)]
    output = features[["__entity_id"]].copy()
    audit = []
    for entry in fields:
        field = entry["field"]
        values = pd.to_numeric(features[field], errors="coerce").replace([np.inf, -np.inf], np.nan)
        missing = values.isna()
        clipped = int(((values < entry["lower"]) | (values > entry["upper"])).sum())
        bounded = values.clip(lower=entry["lower"], upper=entry["upper"])
        output[field + "__scaled"] = (((bounded.fillna(entry["median"]) - entry["median"]) / entry["scale"])
                                       .clip(lower=-5, upper=5).astype(float))
        output[field + "__missing"] = missing.astype(int)
        audit.append({"field": field, "missing_clients": int(missing.sum()),
                      "clipped_to_fit_p01_p99": clipped,
                      "median_on_fit": entry["median"], "scale_on_fit": entry["scale"],
                      "scale_method": entry["scale_method"]})
    if not fields:
        return None, params, {"status": "skipped", "reason": "Нет изменяющихся числовых признаков"}, pd.DataFrame(audit)
    report = {"status": "preview_needs_feature_review", "mode": mode,
              "clients": len(output), "numeric_features_scaled": len(fields),
              "non_numeric_fields_not_encoded": len(features.columns) - 1 - len(numeric_columns),
              "numeric_fields_not_scaled": excluded,
              "missing_values_replaced_by_fit_median": sum(item["missing_clients"] for item in audit),
              "clipped_numeric_values": sum(item["clipped_to_fit_p01_p99"] for item in audit),
              "method": "Ограничение p01..p99, затем (x - median) / IQR и ограничение результата [-5, 5]",
              "warning": "Для проверки по кварталам обучайте параметры только на первом обучающем периоде и применяйте их к следующим; категории и выбор признаков нужны отдельно"}
    return output, params, report, pd.DataFrame(audit)


def build_segmentation_matrix(features, results, combined_report, fitted_recipe=None,
                              max_missing_share=0.5, max_categories=12, exclude_source_fields=()):
    """Create a reproducible numeric matrix; raw feature table stays unchanged."""
    if features is None:
        return None, None, {"status": "skipped", "reason": "Нет таблицы на ID"}, pd.DataFrame()
    linked = set(combined_report.get("linked_tables", []))
    excluded_by_user = set(exclude_source_fields)
    expanded = features.copy()
    candidates_numeric, candidates_category = [], []
    review = []
    warnings = []
    for result in results:
        number = result["table_number"]
        if number not in linked:
            continue
        meta = result["features_report"]
        local_excluded = excluded_by_user | set(meta.get("excluded_source_fields", []))
        prefix = f"__t{number:02d}__"
        status = meta.get("time_status")
        if status not in ("parsed", "declared_timeless"):
            warnings.append(f"Таблица {number}: нет надежного времени; ее признаки исключены из матрицы")
            continue
        if meta.get("id_role_selection") != "manual":
            warnings.append(f"Таблица {number}: роль ID выбрана автоматически, проверьте ее")
        if meta.get("conflicting_rows_excluded", 0):
            warnings.append(f"Таблица {number}: исключены записи с конфликтом ID")
        if status == "declared_timeless":
            warnings.append(f"Таблица {number}: использование данных без времени подтверждено настройкой; проверьте дату среза")
        elif meta.get("as_of_utc"):
            as_of = pd.Timestamp(meta["as_of_utc"])
            for source, target in (("last_observed_at_utc", "recency_days"),
                                   ("first_observed_at_utc", "observed_tenure_days")):
                name = prefix + source
                if name in expanded:
                    date = pd.to_datetime(expanded[name], utc=True, errors="coerce")
                    derived = prefix + target
                    expanded[derived] = (as_of - date).dt.total_seconds().div(86400).clip(lower=0)
                    candidates_numeric.append(derived)
                    review.append({"feature": name, "decision": "derived_days", "reason": derived})
        for field in (prefix + "observed_row_count_30d", prefix + "observed_row_count_90d"):
            if field in expanded:
                candidates_numeric.append(field)
        for field in meta.get("behavior_numeric_features", []):
            if field in expanded:
                candidates_numeric.append(field)
        for source_field, kind in result["inferred_schema"].items():
            base = prefix + str(source_field) + "__"
            if source_field in local_excluded:
                review.append({"feature": base, "decision": "excluded_by_user",
                               "reason": "Поле исключено в EXCLUDE_SOURCE_FIELDS"})
                continue
            if kind == "numeric":
                preferred = meta.get("numeric_aggregate_preferences", {}).get(
                    source_field, "sum" if source_field in meta.get("sum_columns_declared", []) else "median"
                )
                if preferred not in ("median", "min", "max", "latest", "sum"):
                    raise ValueError(f"Недопустимый агрегат {preferred!r} для поля {source_field!r}")
                field = base + preferred
                if field in expanded:
                    candidates_numeric.append(field)
            elif kind == "numeric_discrete":
                field = base + "latest"
                if field in expanded:
                    candidates_numeric.append(field)
                    warnings.append(f"{field}: проверьте, что это счетчик, а не числовой код")
            elif kind == "categorical":
                for suffix, target in (("most_common", candidates_category),
                                       ("distinct_count", candidates_numeric)):
                    field = base + suffix
                    if field in expanded:
                        target.append(field)
            elif kind == "boolean":
                field = base + "latest"
                if field in expanded:
                    candidates_category.append(field)
    candidates_numeric = list(dict.fromkeys(candidates_numeric))
    candidates_category = list(dict.fromkeys(candidates_category))
    if fitted_recipe is not None:
        if (not isinstance(fitted_recipe, dict) or fitted_recipe.get("version") != 1
                or fitted_recipe.get("method") != "selected_robust_numeric_and_one_hot"
                or not isinstance(fitted_recipe.get("numeric_params"), dict)
                or not isinstance(fitted_recipe.get("categories"), list)):
            raise ValueError("NORMALIZATION_PARAMS_PATH: несовместимый формат рецепта")
        numeric_names = [entry["field"] for entry in fitted_recipe["numeric_params"]["fields"]]
        category_specs = fitted_recipe["categories"]
        permitted = set(candidates_numeric + candidates_category)
        missing_required = [name for name in numeric_names + [c["field"] for c in category_specs]
                            if name not in expanded or name not in permitted]
        if missing_required:
            raise ValueError(f"Для применения базовых параметров отсутствуют нужные признаки: {missing_required[:8]}")
        selected_numeric = numeric_names
        recipe = fitted_recipe
        mode = "apply_saved_recipe"
    else:
        selected_numeric = []
        for field in candidates_numeric:
            series = pd.to_numeric(expanded[field], errors="coerce").replace([np.inf, -np.inf], np.nan)
            share = float(series.isna().mean())
            if share > max_missing_share:
                review.append({"feature": field, "decision": "excluded", "reason": "пропусков больше порога"})
            elif series.dropna().nunique() < 2:
                review.append({"feature": field, "decision": "excluded", "reason": "постоянное или пустое поле"})
            else:
                selected_numeric.append(field)
                review.append({"feature": field, "decision": "selected_numeric", "reason": "устойчивое масштабирование"})
        category_specs = []
        for field in candidates_category:
            values = expanded[field].astype("string")
            share = float(values.isna().mean())
            frequency = values.dropna().astype(str).value_counts()
            if share > max_missing_share:
                review.append({"feature": field, "decision": "excluded", "reason": "пропусков больше порога"})
            elif len(frequency) < 2:
                review.append({"feature": field, "decision": "excluded", "reason": "одна категория или пустое поле"})
            else:
                top = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))[:max_categories]
                category_specs.append({"field": field, "levels": [str(value) for value, _ in top]})
                reason = "частые категории + прочие" if len(frequency) > max_categories else "категории в числовом виде"
                review.append({"feature": field, "decision": "selected_category", "reason": reason})
        recipe = None
        mode = "fit_current_input"
    source_numeric = expanded[["__entity_id", *selected_numeric]].copy()
    numeric_matrix, numeric_params, numeric_report, numeric_audit = scale_client_numeric_features(
        source_numeric, None if fitted_recipe is None else fitted_recipe["numeric_params"]
    )
    if numeric_matrix is None:
        numeric_matrix = expanded[["__entity_id"]].copy()
    if fitted_recipe is None:
        recipe = {"version": 1, "method": "selected_robust_numeric_and_one_hot",
                  "numeric_params": numeric_params, "categories": category_specs,
                  "fit_as_of_utc": [r["features_report"].get("as_of_utc")
                                    for r in results if r["table_number"] in linked]}
    result_matrix = numeric_matrix.copy()
    encoded_columns = {}
    unseen = {}
    if len({c["field"] for c in category_specs}) != len(category_specs):
        raise ValueError("Повторяются поля категорий в параметрах нормализации")
    for i, spec in enumerate(category_specs, start=1):
        field, levels = spec["field"], spec["levels"]
        if not isinstance(levels, list) or len(levels) != len(set(levels)) or any(not isinstance(v, str) for v in levels):
            raise ValueError("Некорректные категории в параметрах нормализации")
        values = expanded[field].astype("string")
        group = f"__cat{i:03d}__"
        weight = 1 / math.sqrt(2)
        for j, value in enumerate(levels, start=1):
            encoded_columns[group + f"value_{j:02d}"] = values.eq(value).fillna(False).astype(float) * weight
        other = values.notna() & ~values.isin(levels)
        encoded_columns[group + "other"] = other.fillna(False).astype(float) * weight
        encoded_columns[group + "missing"] = values.isna().astype(float) * weight
        unseen[field] = int(other.sum())
    if encoded_columns:
        result_matrix = pd.concat([result_matrix, pd.DataFrame(encoded_columns, index=result_matrix.index)], axis=1)
    if result_matrix.shape[1] <= 1:
        return None, recipe, {"status": "skipped", "reason": "После отбора нет изменяющихся признаков"}, pd.DataFrame(review)
    if fitted_recipe is None:
        # Keep the fit-period selection fixed for all future quarters. Drop
        # empty indicators and identical encodings that add weight without
        # adding any information to distance-based clustering.
        dropped = []
        kept = []
        fingerprints = {}
        for column in result_matrix.columns:
            if column == "__entity_id":
                continue
            values = result_matrix[column]
            if values.nunique(dropna=False) <= 1:
                dropped.append({"feature": column, "decision": "excluded_constant_matrix_column",
                                "reason": "постоянное значение на обучающей выборке"})
                continue
            fingerprint = tuple(values.to_numpy())
            if fingerprint in fingerprints:
                dropped.append({"feature": column, "decision": "excluded_duplicate_matrix_column",
                                "reason": "точная копия " + fingerprints[fingerprint]})
                continue
            fingerprints[fingerprint] = column
            kept.append(column)
        if not kept:
            return None, recipe, {"status": "skipped", "reason": "Нет различающихся клиентов признаков"}, pd.DataFrame(review + dropped)
        result_matrix = result_matrix[["__entity_id", *kept]]
        recipe["output_columns"] = kept
        review.extend(dropped)
    elif "output_columns" in recipe:
        expected = recipe["output_columns"]
        if (not isinstance(expected, list) or not expected or len(expected) != len(set(expected))
                or any(name not in result_matrix for name in expected)):
            raise ValueError("В сохраненном рецепте отсутствуют выходные признаки матрицы")
        result_matrix = result_matrix[["__entity_id", *expected]]
    assert result_matrix["__entity_id"].is_unique
    assert np.isfinite(result_matrix.drop(columns=["__entity_id"]).to_numpy(dtype=float)).all()
    selected = set(selected_numeric + [c["field"] for c in category_specs])
    for field in features.columns:
        if field != "__entity_id" and field not in selected and not any(x["feature"] == field for x in review):
            review.append({"feature": field, "decision": "not_selected",
                           "reason": "другой агрегат того же поля, ID, дата или источник без надежного времени"})
    if numeric_audit is not None:
        for item in numeric_audit.to_dict(orient="records"):
            review.append({"feature": item["field"], "decision": "normalization", "reason":
                           f"missing={item['missing_clients']}; clipped={item['clipped_to_fit_p01_p99']}; "
                           f"median={item['median_on_fit']}; scale={item['scale_on_fit']}"})
    scaled = result_matrix.filter(regex=r"__scaled$")
    variable = scaled.loc[:, scaled.nunique(dropna=False).gt(1)]
    correlations = []
    if variable.shape[1] > 1:
        corr = variable.corr().abs()
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i + 1:]:
                value = corr.loc[left, right]
                if pd.notna(value) and value >= 0.98:
                    correlations.append({"feature_a": left, "feature_b": right, "abs_pearson": round(float(value), 4)})
    report = {"status": "preview_needs_semantic_review", "mode": mode,
              "clients": len(result_matrix), "matrix_columns_including_id": len(result_matrix.columns),
              "matrix_columns_excluded_constant": sum(r["decision"] == "excluded_constant_matrix_column" for r in review),
              "matrix_columns_excluded_duplicate": sum(r["decision"] == "excluded_duplicate_matrix_column" for r in review),
              "numeric_features_scaled": len(numeric_params["fields"]),
              "categorical_features_encoded": len(category_specs),
              "unseen_category_values": unseen,
              "numeric_scaling": numeric_report,
              "high_numeric_correlations_for_review": correlations[:30],
              "review_warnings": warnings,
              "not_model_ready_until": "Подтвердить ID, время, смысл признаков, источники и период обучения"}
    return result_matrix, recipe, report, pd.DataFrame(review)


def run_generic_pipeline(paths, *, sheets=None, header_rows=None, table_settings=None, type_overrides=None,
                         timezone="UTC", dayfirst=None, date_formats=None, missing_tokens=(), id_column=None,
                         time_column=None, as_of_date=None, windows=(30, 90)):
    sources, inventory = load_tables(paths, sheets, header_rows)
    table_settings = table_settings or {}
    results = []
    for number, (file, sheet, frame, header_row) in enumerate(sources, start=1):
        options = table_settings.get((file, sheet), table_settings.get(file, {}))
        if not isinstance(options, dict):
            raise ValueError(f"TABLE_SETTINGS[{(file, sheet)!r}] должен быть словарем")
        types = {**(type_overrides or {}), **options.get("type_overrides", {})}
        formats = {**(date_formats or {}), **options.get("date_formats", {})}
        local_timezone = options.get("timezone", timezone)
        local_dayfirst = options.get("dayfirst", dayfirst)
        local_id = options.get("id_column", id_column)
        local_time = options.get("time_column", time_column)
        unused = sorted(str(k) for k in set(types) | set(formats) if k not in frame.columns)
        types = {k: v for k, v in types.items() if k in frame.columns}
        formats = {k: v for k, v in formats.items() if k in frame.columns}
        prepared, quality, issues, schema = prepare_table(
            frame, type_overrides=types, timezone=local_timezone, dayfirst=local_dayfirst,
            date_formats=formats, missing_tokens=options.get("missing_tokens", missing_tokens),
            header_row=header_row
        )
        analysis_view, conversions = build_analysis_view(
            prepared, schema, timezone=local_timezone, dayfirst=local_dayfirst, date_formats=formats
        )
        key_issues, key_report = check_unique_record_key(
            prepared, options.get("unique_key_columns"), header_row=header_row, schema=schema
        )
        issues = pd.concat([issues, key_issues], ignore_index=True)
        entity_id, event_time = infer_history_roles(prepared, schema, local_id, local_time)
        client_index, index_report = build_client_index(
            prepared, entity_id, event_time, options.get("as_of_date", as_of_date), options.get("windows", windows),
            record_key_report=key_report,
        )
        index_report["id_role_selection"] = "manual" if local_id else ("heuristic_needs_review" if entity_id else "undetermined")
        index_report["time_role_selection"] = "manual" if local_time else ("heuristic_needs_review" if event_time else "undetermined")
        results.append({
            "file": file, "sheet": sheet, "prepared": prepared, "analysis_ready": analysis_view,
            "analysis_conversions": conversions, "column_quality": quality, "row_issues": issues,
            "inferred_schema": schema, "client_index": client_index, "index_report": index_report,
            "record_key_report": key_report,
            "_feature_options": {"entity_id": entity_id, "event_time": event_time, "header_row": header_row,
                                 "as_of_date": options.get("as_of_date", as_of_date),
                                 "windows": options.get("windows", windows),
                                 "sum_columns": options.get("sum_columns", ()),
                                 "behavior_rollups": options.get("behavior_rollups", ()),
                                 "exclude_source_fields": options.get("exclude_source_fields", ()),
                                 "numeric_aggregate_preferences": options.get("numeric_aggregate_preferences", {}),
                                 "allow_timeless_features": options.get("allow_timeless_features", False)},
            "summary": {
                "input_rows": len(frame), "prepared_rows": len(prepared),
                "duplicate_full_rows_flagged": int(prepared.duplicated(keep=False).sum()),
                "parse_errors": int(quality["parse_failures"].sum()),
                "original_field_names_preserved": list(prepared.columns) == list(frame.columns),
                "review_fields": int(quality["review_flags"].map(bool).sum()),
                "declared_missing_markers_normalized": int(issues["issue"].eq("declared_missing_marker_normalized").sum()),
                "possible_missing_marker_cells": int(sum(sum(markers.values()) for markers in quality["possible_missing_markers"])),
                "record_key_conflict_rows": key_report["conflict_rows"],
                "record_key_suspected_conflict_rows": key_report["suspected_conflict_rows"],
                "record_key_uncertain_repeat_rows": key_report["uncertain_repeat_rows"],
                "record_key_exact_repeat_rows": key_report["exact_repeat_rows"],
                "analysis_invalid_cells_set_missing": sum(item["invalid_as_missing"] for item in conversions),
                "analysis_case_normalized_cells": sum(item["case_normalized_cells"] for item in conversions),
                "unused_type_or_date_overrides": unused,
            },
            "table_number": number,
        })
    reference_ids = [(r["_feature_options"]["entity_id"], set(r["analysis_ready"][r["_feature_options"]["entity_id"]]
                         .dropna().map(_as_text))) for r in results if r["_feature_options"]["entity_id"]]
    for result in results:
        settings = result.pop("_feature_options")
        entity_id = settings["entity_id"]
        selection = "manual" if ((table_settings.get((result["file"], result["sheet"]),
                                    table_settings.get(result["file"], {})) or {}).get("id_column", id_column)) else "heuristic_needs_review"
        if entity_id is None:
            matches = []
            for field, kind in result["inferred_schema"].items():
                if kind != "identifier":
                    continue
                values = set(result["analysis_ready"][field].dropna().map(_as_text))
                if len(values) < 10:
                    continue
                if any(ref_field == field and len(values & ref_values) / max(1, min(len(values), len(ref_values))) >= 0.8
                       for ref_field, ref_values in reference_ids):
                    matches.append(field)
            if len(matches) == 1:
                entity_id = matches[0]
                selection = "cross_table_heuristic_needs_review"
        features, features_report = build_client_features(
            result["analysis_ready"], result["inferred_schema"], result["row_issues"],
            table_number=result["table_number"], header_row=settings["header_row"], id_column=entity_id,
            time_column=settings["event_time"], as_of_date=settings["as_of_date"],
            windows=settings["windows"], sum_columns=settings["sum_columns"],
            behavior_rollups=settings["behavior_rollups"],
            allow_timeless_features=settings["allow_timeless_features"],
        )
        features_report["id_role_selection"] = selection if entity_id else "undetermined"
        features_report["excluded_source_fields"] = list(settings["exclude_source_fields"])
        features_report["numeric_aggregate_preferences"] = settings["numeric_aggregate_preferences"]
        result["client_features"] = features
        result["features_report"] = features_report
    return results, inventory


## 6. Запуск и просмотр диагностики

Для каждой таблицы отдельно выводится исходное число строк, обнаруженные типы, ошибки преобразования, сведения об истории и производной таблице признаков. `conversion_applied` относится к `prepared_XX.csv`, а `analysis_conversions` отдельно описывает преобразования в аналитической копии. Значение `False` в исходном контроле означает, что столбец не был целиком преобразован: его валидные ячейки все равно могут быть разобраны в аналитической копии.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

results, source_inventory = run_generic_pipeline(
    INPUT_PATHS,
    sheets=SHEETS,
    header_rows=HEADER_ROWS,
    table_settings=TABLE_SETTINGS,
    type_overrides=TYPE_OVERRIDES,
    timezone=SOURCE_TIMEZONE,
    dayfirst=DAYFIRST,
    date_formats=DATE_FORMATS,
    missing_tokens=MISSING_TOKENS,
    id_column=ID_COLUMN,
    time_column=TIME_COLUMN,
    as_of_date=AS_OF_DATE,
    windows=WINDOWS_DAYS,
)

print("Обработанные таблицы:")
display(pd.DataFrame(source_inventory))
for result in results:
    print("\nТаблица", result["table_number"], "—", result["file"], "/", result["sheet"] or "CSV/TSV")
    print("Проверка:", result["summary"])
    print("Ключ записи:", result["record_key_report"])
    print("История:", result["index_report"])
    print("Признаки на ID:", result["features_report"])
    changed = [item for item in result["analysis_conversions"] if item["status"] != "unchanged"]
    print("Изменения в аналитической копии:", changed)
    display(result["column_quality"][["field", "source_dtype", "inferred_type", "missing_count", "parse_failures", "conversion_applied", "review_flags"]])
combined_features, combined_report = combine_client_features(results)
print("Общая таблица на ID:", combined_report)


## 7. Нормализация и матрица для сегментации

Вход — `client_features_combined.csv`, по одной строке на ID. Из числовых исходных полей берется один согласованный агрегат: медиана, а сумма — только если она явно разрешена; `numeric_aggregate_preferences` позволяет выбрать последнее значение, например для текущего остатка; для счетчиков берется последнее значение. Подтвержденные поведенческие расчеты из `behavior_rollups` добавляются отдельно. Для категорий берется частое значение и число разных значений. Даты переводятся в давность относительно даты среза. Из одного поля не включаются одновременно медиана, минимум, максимум и последнее значение. Постоянные и преимущественно пустые поля исключаются с причиной в `feature_selection.csv`.

Числа ограничиваются по 1-му и 99-му процентилям обучающей выборки, пустые значения заполняются ее медианой, затем масштабируются через медиану и межквартильный размах. Добавляется индикатор пропуска. Категории кодируются отдельными числовыми столбцами: до 12 частых значений, «прочие» и «пропуск». Постоянные и точные дубликаты числовых столбцов удаляются из матрицы, решение сохраняется для последующих срезов. Все параметры сохраняются для применения на новых периодах. Исходные значения остаются в других файлах архива.

Матрица годится для **первого эксперимента сегментации**, после проверки предупреждений и смысла полей. Генерация признаков из категорий не определяет, какие именно закономерности нужны бизнесу, а оценку качества и устойчивости сегментов нужно делать отдельно.


In [ ]:
normalization_params = None
if NORMALIZATION_PARAMS_PATH:
    with open(NORMALIZATION_PARAMS_PATH, "r", encoding="utf-8") as fh:
        normalization_params = json.load(fh)

segmentation_matrix, normalization_recipe, normalization_report, feature_selection = build_segmentation_matrix(
    combined_features, results, combined_report,
    fitted_recipe=normalization_params,
    max_missing_share=MAX_MISSING_SHARE,
    max_categories=MAX_CATEGORIES,
    exclude_source_fields=EXCLUDE_SOURCE_FIELDS,
)
print("Матрица сегментации:", {k: v for k, v in normalization_report.items() if k != "numeric_scaling"})
if segmentation_matrix is not None:
    print("Форма матрицы:", segmentation_matrix.shape)
    display(feature_selection[feature_selection["decision"].eq("excluded")].head(20))


## 8. Проверка и скачивание результатов

Архив содержит `prepared_XX.csv` с проверяемыми значениями и `analysis_ready_XX.csv` для анализа; `column_quality_XX.csv`, `row_issues_XX.csv` и, где удалось, `client_index_XX.csv`, `client_features_XX.csv`. Если источники удалось технически связать, добавляется `client_features_combined.csv`.

Для сегментации используются `segmentation_matrix.csv` (только числовые признаки и ID), `feature_selection.csv` (что вошло и почему), `normalization_params.json` (параметры обучающего периода). `run_report.json` хранит источники, типы, причины пропуска, преобразования, исключения при сборке признаков и способ выбора ID/времени. Номера `XX` соответствуют таблицам из блока выше. Данные еще не проверены реальными сегментами или квартальным тестом.


In [ ]:
import json
import zipfile

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = []
report = {"sources": source_inventory, "tables": []}
for result in results:
    number = f"{result['table_number']:02d}"
    outputs = {
        f"prepared_{number}.csv": result["prepared"],
        f"analysis_ready_{number}.csv": result["analysis_ready"],
        f"column_quality_{number}.csv": result["column_quality"],
        f"row_issues_{number}.csv": result["row_issues"],
    }
    if result["client_index"] is not None:
        outputs[f"client_index_{number}.csv"] = result["client_index"]
    if result["client_features"] is not None:
        outputs[f"client_features_{number}.csv"] = result["client_features"]
    for filename, frame in outputs.items():
        destination = OUTPUT_DIR / filename
        frame.to_csv(destination, index=False, encoding="utf-8-sig")
        artifacts.append(destination)
    report["tables"].append({
        "file": result["file"], "sheet": result["sheet"] or None,
        "summary": result["summary"], "inferred_schema": result["inferred_schema"],
        "client_index": result["index_report"],
        "client_features": result["features_report"],
        "analysis_conversions": result["analysis_conversions"],
        "record_key": result["record_key_report"],
        "field_quality": json.loads(result["column_quality"].to_json(orient="records", force_ascii=False)),
    })

report["combined_client_features"] = combined_report
if combined_features is not None:
    destination = OUTPUT_DIR / "client_features_combined.csv"
    combined_features.to_csv(destination, index=False, encoding="utf-8-sig")
    artifacts.append(destination)
report["segmentation_matrix"] = normalization_report
if segmentation_matrix is not None:
    destination = OUTPUT_DIR / "segmentation_matrix.csv"
    segmentation_matrix.to_csv(destination, index=False, encoding="utf-8-sig")
    artifacts.append(destination)
if normalization_recipe is not None:
    destination = OUTPUT_DIR / "normalization_params.json"
    destination.write_text(json.dumps(normalization_recipe, ensure_ascii=False, indent=2), encoding="utf-8")
    artifacts.append(destination)
if not feature_selection.empty:
    destination = OUTPUT_DIR / "feature_selection.csv"
    feature_selection.to_csv(destination, index=False, encoding="utf-8-sig")
    artifacts.append(destination)

report_path = OUTPUT_DIR / "run_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
artifacts.append(report_path)

archive = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in artifacts:
        zf.write(path, arcname=path.name)

print("Готово:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive))
